In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, train_test_split

In [2]:
df = pd.read_csv("course_lead_scoring_2026.csv")
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [5]:
cat_cols = df.select_dtypes(include=['object', 'category', 'str']).columns
num_cols = df.select_dtypes(include=['number']).columns

df[cat_cols] = df[cat_cols].fillna('NA')
df[num_cols] = df[num_cols].fillna(0.0)

In [6]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_full_train = df_full_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_full_train = df_full_train['converted'].values
y_test = df_test['converted'].values

del df_train['converted']
del df_val['converted']
del df_full_train['converted']
del df_test['converted']

In [7]:
q1_features = [
    'lead_score',
    'number_of_courses_viewed',
    'interaction_count',
    'annual_income',
]

auc_scores = {}
for col in q1_features:
    auc = roc_auc_score(y_train, df_train[col])
    if auc < 0.5:
        auc = roc_auc_score(y_train, -df_train[col])
    auc_scores[col] = auc
    print(f'{col}: {auc:.4f}')

best_feature = max(auc_scores, key=auc_scores.get)
print(f'\nHighest AUC feature: {best_feature}')

lead_score: 0.7882
number_of_courses_viewed: 0.7230
interaction_count: 0.7649
annual_income: 0.6082

Highest AUC feature: lead_score


In [8]:
dv = DictVectorizer(sparse=False)

train_dicts = df_train.to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val.to_dict(orient='records')
X_val = dv.transform(val_dicts)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]
val_auc = roc_auc_score(y_val, y_pred)

print(f'Validation AUC: {val_auc:.3f}')

Validation AUC: 0.732


In [10]:
thresholds = np.arange(0.0, 1.01, 0.01)
precisions = []
recalls = []

for t in thresholds:
    tp = ((y_pred >= t) & (y_val == 1)).sum()
    fp = ((y_pred >= t) & (y_val == 0)).sum()
    fn = ((y_pred < t) & (y_val == 1)).sum()

    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0

    precisions.append(p)
    recalls.append(r)

diffs = [
    (abs(p - r), t)
    for p, r, t in zip(precisions, recalls, thresholds)
    if p > 0 and r > 0
]
best_threshold = min(diffs, key=lambda x: x[0])[1]

print(f'Intersection threshold: {best_threshold:.2f}')

Intersection threshold: 0.63


In [11]:
f1_scores = []

for p, r in zip(precisions, recalls):
    if p + r > 0:
        f1 = 2 * (p * r) / (p + r)
    else:
        f1 = 0
    f1_scores.append(f1)

best_f1_idx = np.argmax(f1_scores)
print(f'Max F1 Score: {f1_scores[best_f1_idx]:.4f}')
print(f'Threshold for Max F1: {thresholds[best_f1_idx]:.2f}')

Max F1 Score: 0.7576
Threshold for Max F1: 0.41


In [12]:
kf = KFold(n_splits=5, shuffle=True, random_state=1)
scores = []

for train_idx, val_idx in kf.split(df_full_train):
    df_fold_train = df_full_train.iloc[train_idx]
    df_fold_val = df_full_train.iloc[val_idx]

    y_fold_train = y_full_train[train_idx]
    y_fold_val = y_full_train[val_idx]

    dv_fold = DictVectorizer(sparse=False)
    X_fold_train = dv_fold.fit_transform(
        df_fold_train.to_dict(orient='records')
    )
    X_fold_val = dv_fold.transform(df_fold_val.to_dict(orient='records'))

    model_fold = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)
    model_fold.fit(X_fold_train, y_fold_train)

    preds = model_fold.predict_proba(X_fold_val)[:, 1]
    scores.append(roc_auc_score(y_fold_val, preds))

print(f'Mean AUC: {np.mean(scores):.3f}')
print(f'Standard Deviation: {np.std(scores):.3f}')

Mean AUC: 0.732
Standard Deviation: 0.007


In [13]:
c_values = [0.000001, 0.001, 1]
results = []

for C in c_values:
    kf = KFold(n_splits=5, shuffle=True, random_state=1)
    fold_scores = []

    for train_idx, val_idx in kf.split(df_full_train):
        df_fold_train = df_full_train.iloc[train_idx]
        df_fold_val = df_full_train.iloc[val_idx]

        y_fold_train = y_full_train[train_idx]
        y_fold_val = y_full_train[val_idx]

        dv_fold = DictVectorizer(sparse=False)
        X_fold_train = dv_fold.fit_transform(
            df_fold_train.to_dict(orient='records')
        )
        X_fold_val = dv_fold.transform(df_fold_val.to_dict(orient='records'))

        model_tune = LogisticRegression(solver='liblinear', C=C, max_iter=1000)
        model_tune.fit(X_fold_train, y_fold_train)

        preds = model_tune.predict_proba(X_fold_val)[:, 1]
        fold_scores.append(roc_auc_score(y_fold_val, preds))

    mean_score = round(float(np.mean(fold_scores)), 3)
    std_score = round(float(np.std(fold_scores)), 3)
    results.append({'C': C, 'mean': mean_score, 'std': std_score})
    print(f'C={C:<8} | Mean AUC: {mean_score:.3f} | Std: {std_score:.3f}')

best_result = sorted(results, key=lambda x: (-x['mean'], x['std'], x['C']))[0]
print(f"\nBest parameter C: {best_result['C']}")

C=1e-06    | Mean AUC: 0.617 | Std: 0.019
C=0.001    | Mean AUC: 0.749 | Std: 0.010
C=1        | Mean AUC: 0.732 | Std: 0.007

Best parameter C: 0.001
